# Measures and Aggregations

In SLayer, a **measure** is a named row-level SQL expression. The **aggregation** (sum, avg, count, etc.) is specified at query time as a function call: `sum(revenue)`, `count(*)`, `weighted_avg(tax_rate, weight=subtotal)`.

This notebook demonstrates the full range of aggregation features with working code.

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Basic Aggregations

Every field formula specifies both the measure and the aggregation as a function call. `count(*)` is the universal row count.

In [2]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(order_total)", "avg(order_total)", "min(order_total)", "max(order_total)"],
    }
)

pd.DataFrame(result.data)

,orders._count,orders.order_total_sum,orders.order_total_avg,orders.order_total_min,orders.order_total_max
0,661186,7405282.54,11.20,0.00,106.42


## Inspecting Model Measures

Auto-ingested models have one column per underlying table column - no need for num_columns x num_aggregations predefined measures, those can be done on the fly. 

In [3]:
orders_model = next(m for m in models if m.name == "orders")

print(f"orders_model has {len(orders_model.columns)} columns:")
for col in orders_model.columns:
    sql_str = col.sql or col.name
    print(f"  {col.name:<20} sql: {sql_str:<20} type: {col.type}")

print("\nAny column can be aggregated at query time via a function call,")
print("subject to the column's type and allowed_aggregations whitelist.")
print("A sole primary key defaults to count / count_distinct / min / max only.")
print("Plus count(*) is always available for COUNT(*).")


orders_model has 9 columns:
  id                   sql: id                   type: TEXT
  customer_id          sql: customer_id          type: TEXT
  ordered_at           sql: ordered_at           type: DATE
  store_id             sql: store_id             type: TEXT
  subtotal             sql: subtotal             type: DOUBLE
  tax_paid             sql: tax_paid             type: DOUBLE
  order_total          sql: order_total          type: DOUBLE
  status               sql: status               type: TEXT
  item_skus            sql: item_skus            type: ARRAY

Any column can be aggregated at query time via a function call,
subject to the column's type and allowed_aggregations whitelist.
A sole primary key defaults to count / count_distinct / min / max only.
Plus count(*) is always available for COUNT(*).


## Multiple Aggregations on One Measure

One measure, many aggregations — no need to define `order_total_sum`, `order_total_avg`, etc. as separate measures.

In [4]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(order_total)", "avg(order_total)"],
        "dimensions": ["stores.name"],
        "order": [{"column": "sum(order_total)", "direction": "desc"}],
        "limit": 5,
    }
)

pd.DataFrame(result.data)

,orders.stores.name,orders._count,orders.order_total_sum,orders.order_total_avg
0,Brooklyn,255457,2786150.77,10.91
1,Philadelphia,192311,2159496.69,11.23
2,Chicago,104242,1206889.67,11.58
3,San Francisco,93449,1079032.56,11.55
4,New Orleans,15727,173712.85,11.05


## Arithmetic on Aggregated Measures

Aggregated measures compose naturally in arithmetic expressions.

In [5]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "count(*)",
            "sum(order_total)",
            {"formula": "sum(order_total) / count(*)", "name": "aov", "label": "Avg Order Value"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"column": "sum(order_total)", "direction": "desc"}],
        "limit": 5,
    }
)

pd.DataFrame(result.data)

,orders.stores.name,orders._count,orders.order_total_sum,orders.aov
0,Brooklyn,255457,2786150.77,10.91
1,Philadelphia,192311,2159496.69,11.23
2,Chicago,104242,1206889.67,11.58
3,San Francisco,93449,1079032.56,11.55
4,New Orleans,15727,173712.85,11.05


## Share of Parent (`partition_by`)

Most aggregations take an optional `partition_by=` to compute over a **subset** of the query's dimensions, repeated across the finer rows — like `SUM(x) OVER (PARTITION BY ...)`. `partition_by=[]` is the grand total (share of total); a dimension, list, or dotted path gives share-of-parent within a group. The total is computed over rows passing row-level filters, so `having`-style filters and pagination never change it.

It composes with `window=`, `first`/`last`, transforms, and filter / `order` references — cross-model included, e.g. `last(customers.spend, partition_by=customers.tier)` or a cross-model partitioned aggregate nested inside a transform.

In [6]:
# `partition_by=` computes an aggregation at a COARSER grain than the query and
# repeats it across the finer rows. `partition_by=[]` is the grand total, so
# each store's share of total revenue sums to 100%. A dimension, list, or dotted
# path (e.g. partition_by=region) gives share-of-parent within a group.
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "dimensions": ["stores.name"],
        "measures": [
            {"formula": "sum(order_total)", "name": "store_rev"},
            {"formula": "sum(order_total) / sum(order_total, partition_by=[])", "name": "share_of_total"},
        ],
        "order": [{"column": "store_rev", "direction": "desc"}],
    }
)

df = pd.DataFrame(result.data)
print(f"sum of shares: {df['orders.share_of_total'].sum():.1%}")
df

sum of shares: 100.0%


,orders.stores.name,orders.store_rev,orders.share_of_total
0,Brooklyn,2786150.77,0.38
1,Philadelphia,2159496.69,0.29
2,Chicago,1206889.67,0.16
3,San Francisco,1079032.56,0.15
4,New Orleans,173712.85,0.02


## Expression Dimensions and Conditionals (`CASE WHEN` / `iif`)

Expressions for measures, filters, and dimensions can branch with SQL `CASE WHEN c THEN x [WHEN ...] [ELSE y] END` — or the equivalent call form `iif(c, x, y)` — rendered as portable SQL `CASE` on every backend. Branch types must agree: a `NULL` branch (a missing `ELSE`) is absorbed by the other, a numeric mix widens to `DOUBLE`, and any other mix is a plan-time error.

A dimension can itself be a computed expression: pass `{"expression": ..., "name": ...}` (or just a non-identifier string) in `dimensions` and the query groups by the computed value. Its name works in `order` and `filters` like any other dimension.

In [7]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "dimensions": [
            {
                "expression": "CASE WHEN order_total >= 15 THEN 'large' ELSE 'small' END",
                "name": "size",
            },
        ],
        "measures": [
            {"formula": "count(*)", "name": "n_orders"},
            {"formula": "sum(order_total)", "name": "revenue"},
            {"formula": "avg(order_total)", "name": "average_order_size"},
        ],
        "order": [{"column": "revenue", "direction": "desc"}],
    }
)

pd.DataFrame(result.data)

,orders.size,orders.n_orders,orders.revenue,orders.average_order_size
0,large,153292,4204839.08,27.43
1,small,507894,3200443.46,6.30


## Transforms on Aggregated Measures

All transform functions (`cumsum`, `change`, `time_shift`, `rank`, etc.) wrap aggregated measure refs.

In [8]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "sum(order_total)",
            {"formula": "cumsum(sum(order_total))", "name": "running_total"},
            {"formula": "change(sum(order_total))", "name": "mom_change"},
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

pd.DataFrame(result.data)

,orders.ordered_at,orders.order_total_sum,orders.running_total,orders.mom_change
0,2023-11-01,14258.51,14258.51,None
1,2023-12-01,24851.44,39109.95,10592.93
2,2024-01-01,30322.85,69432.80,5471.41
3,2024-02-01,36805.46,106238.26,6482.61
4,2024-03-01,45741.81,151980.07,8936.35
5,2024-04-01,47484.22,199464.29,1742.41


## Cross-Model Aggregated Measures

Cross-model measures use dot syntax for the join path, wrapped in the aggregation. `count(customers.*)` means COUNT(*) on the joined customers model.

In [9]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "count(*)",
            "sum(order_total)",
            "count(customers.*)",
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

pd.DataFrame(result.data)

,orders.ordered_at,orders._count,orders.order_total_sum,orders.customers._count
0,2023-11-01,1338,14258.51,2583
1,2023-12-01,2286,24851.44,2583
2,2024-01-01,2641,30322.85,2583
3,2024-02-01,3276,36805.46,2583
4,2024-03-01,4149,45741.81,2583
5,2024-04-01,4171,47484.22,2583


## Custom Aggregation: Weighted Average

Let's define a model with a custom weighted average aggregation.

In [10]:
from slayer.core.enums import DataType
from slayer.core.models import Aggregation, AggregationParam, Column, ModelJoin, SlayerModel

orders_custom = SlayerModel(
    name="orders_custom",
    sql_table="orders",
    data_source="jaffle_shop",
    columns=[
        Column(name="id", sql="id", type=DataType.TEXT, primary_key=True),
        Column(name="store_id", sql="store_id", type=DataType.TEXT),
        Column(name="subtotal", sql="subtotal", type=DataType.DOUBLE),
        Column(name="store_tax_rate", sql="stores.tax_rate", type=DataType.DOUBLE),
    ],
    joins=[ModelJoin(target_model="stores", join_pairs=[["store_id", "id"]])],
    aggregations=[
        Aggregation(
            name="weighted_avg",
            formula="SUM({value} * {weight}) / NULLIF(SUM({weight}), 0)",
            params=[AggregationParam(name="weight", sql="subtotal")], # sets the default for the weight parameter
            description="Average of value weighted by weight",
        ),
    ],
)
print(f"Custom aggregations on orders_custom: {[a.name for a in orders_custom.aggregations]}")


Custom aggregations on orders_custom: ['weighted_avg']


## Allowed Aggregations Whitelist

Restrict which aggregations make sense for a given measure.

In [11]:
# Restrict per-column aggregation eligibility via `allowed_aggregations`. The
# whitelist overrides the type-default eligibility rule (numeric columns get
# sum/avg/min/max/etc by default; PK columns are always count-only).
from slayer.core.enums import DataType
from slayer.core.models import Column, SlayerModel

restricted_model = SlayerModel(
    name="restricted_orders",
    sql_table="orders",
    data_source="jaffle_shop",
    columns=[
        Column(name="store_id", sql="store_id", type=DataType.TEXT),
        Column(
            name="subtotal", sql="subtotal", type=DataType.DOUBLE,
            allowed_aggregations=["sum", "avg"],
        ),
    ],
)
sub_col = next(c for c in restricted_model.columns if c.name == "subtotal")
print(f"subtotal column allows: {sub_col.allowed_aggregations}")


subtotal column allows: ['sum', 'avg']


## Generated SQL

The aggregation syntax compiles to standard SQL aggregation functions.

In [12]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(order_total)", "avg(order_total)"],
        "dimensions": ["stores.name"],
        "limit": 3,
    }
)

print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  stores.name AS "orders.stores.name",
  COUNT(*) AS "orders._count",
  SUM(orders.order_total) AS "orders.order_total_sum",
  AVG(orders.order_total) AS "orders.order_total_avg"
FROM orders AS orders
LEFT JOIN stores AS stores
  ON orders.store_id = stores.id
GROUP BY
  stores.name
ORDER BY
  "orders.stores.name" ASC
LIMIT 3


## Summary

| Concept | Syntax | Example |
|---------|--------|---------|
| Aggregation at query time | `agg(measure)` | `sum(revenue)`, `avg(price)` |
| Row count | `count(*)` | Always available, no measure needed |
| Non-null count | `count(col)` | `count(email)` |
| Distinct count | `count_distinct(col)` | `count_distinct(customer_id)` |
| First/last by time | `first(col)`, `last(col)` | `last(balance, updated_at)` |
| Weighted average | `weighted_avg(col, weight=w)` | `weighted_avg(tax_rate, weight=subtotal)` |
| Percentile | `percentile(col, p=0.95)` | `percentile(latency, p=0.99)` |
| Median | `median(col)` | Shorthand for percentile(p=0.5) |
| Share of parent | `agg(col, partition_by=...)` | `sum(revenue, partition_by=region)`, `partition_by=[]` for total |
| Custom formula | Define in model `aggregations` | `trimmed_mean(score, lo=10, hi=90)` |
| Whitelist | `allowed_aggregations` on measure | Restricts valid aggregations |
| Conditional | `CASE WHEN ... END` / `iif(c, x, y)` | `CASE WHEN sum(revenue) >= 1000 THEN 'hi' ELSE 'lo' END` |
| Expression dimension | `{"expression": ..., "name": ...}` in `dimensions` | Groups by the computed value |

See the [aggregations guide](aggregations.md) for the full explanation.